# OCR Comparison — Marker vs Chandra
**Yêu cầu:** Runtime → Change runtime type → **T4 GPU** (Colab Pro)

Notebook này test Marker và Chandra trên cùng một ảnh để so sánh.

In [ ]:
# ── 0. Kiểm tra GPU ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())

In [ ]:
# ── 1. Cài dependencies ───────────────────────────────────────────────────────
# ⚠️  CONFLICT: Marker vs Chandra cần 2 phiên bản transformers khác nhau:
#   • Marker  → surya-ocr 0.17.x dùng transformers.onnx → cần transformers <5.0
#   • Chandra → model qwen3_5 mới → cần transformers ≥5.x
#
# Notebook này ưu tiên MARKER (pin <5.x).
# Để test CHANDRA riêng → dùng: notebooks/test_chandra_colab.ipynb

!pip install -q 'transformers>=4.45.2,<5.0.0'
!pip install -q marker-pdf
!pip install -q bitsandbytes pymupdf
# Pin lại lần cuối phòng marker-pdf kéo transformers lên 5.x
!pip install -q 'transformers>=4.45.2,<5.0.0'
print('✅ Cài xong (transformers 4.x, chỉ dùng được Marker trong kernel này).')

In [ ]:
# ── 2. Upload file (ảnh hoặc PDF) ────────────────────────────────────────────
from google.colab import files
import io, os
from PIL import Image
import fitz  # PyMuPDF

uploaded = files.upload()
fname = list(uploaded.keys())[0]
fdata = uploaded[fname]
ext = os.path.splitext(fname)[1].lower()

is_pdf = (ext == '.pdf')

if is_pdf:
    pdf_path = '/content/test_doc.pdf'
    with open(pdf_path, 'wb') as f:
        f.write(fdata)

    # Render từng trang → PIL (cho Chandra và preview)
    doc = fitz.open(pdf_path)
    pil_pages = []
    for page in doc:
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 144 DPI
        pil_pages.append(Image.frombytes('RGB', [pix.w, pix.h], pix.samples))
    doc.close()

    pil_image = pil_pages[0]  # alias trang đầu cho preview
    print(f'✅ Đã upload PDF: {fname}  →  {len(pil_pages)} trang')
    display(pil_image)
else:
    pdf_path = None
    pil_image = Image.open(io.BytesIO(fdata)).convert('RGB')
    pil_image.save('/content/test_image.png')
    pil_pages = [pil_image]
    print(f'✅ Đã upload ảnh: {fname}  →  {pil_image.size}')
    display(pil_image)

---
## 🔵 Marker OCR

In [ ]:
import os, time
os.environ['TORCH_DEVICE'] = 'cuda'  # GPU mode

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered

print('Loading Marker models...')
t0 = time.time()
converter = PdfConverter(artifact_dict=create_model_dict())
print(f'Load xong: {time.time()-t0:.1f}s')

In [ ]:
import time

t0 = time.time()

if is_pdf:
    # Marker xử lý PDF natively — tốt hơn qua ảnh
    rendered = converter(pdf_path)
else:
    rendered = converter('/content/test_image.png')

marker_text, _, _ = text_from_rendered(rendered)
marker_text = marker_text.strip()
print(f'Marker OCR — {time.time()-t0:.1f}s  ({len(pil_pages)} trang)')
print('─' * 60)
print(marker_text)

In [ ]:
# Hiển thị Markdown đẹp
from IPython.display import Markdown
Markdown(marker_text)

---
## 🟢 Chandra OCR 2

In [ ]:
# Giải phóng VRAM của Marker trước khi load Chandra
import torch, gc
del converter
gc.collect()
torch.cuda.empty_cache()

free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f'VRAM trống: {free_gb:.1f} GB')

In [ ]:
import os
os.environ['TORCH_DEVICE'] = 'cuda'

# Chọn chế độ load dựa trên VRAM còn trống
free_mb = torch.cuda.mem_get_info()[0] // (1024**2)
use_4bit = free_mb < 10000  # < 10GB → dùng 4-bit
print(f'VRAM trống: {free_mb}MB → {"4-bit NF4" if use_4bit else "fp16"}')

from chandra.model import InferenceManager

if use_4bit:
    from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
    import chandra.model as _cm
    from chandra.model import settings as _cs

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        llm_int8_enable_fp32_cpu_offload=True,
    )

    # Patch Params4bit cho transformers 5.x
    try:
        from bitsandbytes.nn.modules import Params4bit
        _orig = Params4bit.__new__
        def _new(cls, *args, **kwargs):
            kwargs.pop('_is_hf_initialized', None)
            return _orig(cls, *args, **kwargs)
        Params4bit.__new__ = staticmethod(_new)
    except Exception:
        pass

    def _load_4bit():
        model = AutoModelForImageTextToText.from_pretrained(
            _cs.MODEL_CHECKPOINT,
            quantization_config=bnb,
            device_map='auto',
            max_memory={0: f'{free_mb-500}MiB', 'cpu': '48GiB'},
            trust_remote_code=True,
        ).eval()
        processor = AutoProcessor.from_pretrained(_cs.MODEL_CHECKPOINT)
        processor.tokenizer.padding_side = 'left'
        model.processor = processor
        return model

    _orig_load = _cm.load_model
    _cm.load_model = _load_4bit
    print('Loading Chandra 4-bit...')
    t0 = time.time()
    manager = InferenceManager(method='hf')
    _cm.load_model = _orig_load
else:
    print('Loading Chandra fp16...')
    t0 = time.time()
    manager = InferenceManager(method='hf')

print(f'Load xong: {time.time()-t0:.1f}s')

In [ ]:
from chandra.model.schema import BatchInputItem

all_md = []
total_t = 0
for i, page_img in enumerate(pil_pages):
    if len(pil_pages) > 1:
        print(f'Chandra: trang {i+1}/{len(pil_pages)}...')
    t0 = time.time()
    batch = [BatchInputItem(image=page_img, prompt_type='ocr_layout')]
    results = manager.generate(batch)
    page_md = getattr(results[0], 'markdown', '') or ''
    all_md.append(page_md)
    elapsed = time.time() - t0
    total_t += elapsed
    if len(pil_pages) > 1:
        print(f'  → {elapsed:.1f}s, {len(page_md)} ký tự')

chandra_md = '\n\n---\n\n'.join(all_md).strip()
print(f'Chandra OCR — tổng {total_t:.1f}s ({len(pil_pages)} trang)')
print('─' * 60)
print(chandra_md[:1000], '...' if len(chandra_md) > 1000 else '')

In [ ]:
Markdown(chandra_md)

---
## 📊 So sánh

In [ ]:
print('=== MARKER ===')
print(f'Ký tự: {len(marker_text)}')
print(marker_text[:500], '...' if len(marker_text) > 500 else '')
print()
print('=== CHANDRA ===')
print(f'Ký tự: {len(chandra_md)}')
print(chandra_md[:500], '...' if len(chandra_md) > 500 else '')